In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load processed data

In [ ]:
df = pd.read_csv('../../data/processed/processed_data.csv')

# Drop Unused columns for the dataset

In [ ]:
# Time based columns as they are arbitrary as time grows and time_since_last_transactions captures better meaning
df.drop(columns=["customer_prev_step", "customer_prev_merchant_step", "merchant_prev_step"], inplace=True)

# Sums and averages are captured in the ratios and z-scores, so can drop them to reduce dimensionality
df.drop(columns=["customer_amount_sum", "customer_avg_amount", "merchant_amount_sum", "merchant_avg_amount", "global_amount_sum"], inplace=True)

# Was only used to dervive fraud rate, can drop it now
df.drop(columns=["merchant_fraud_count"], inplace=True)

# Dropping from data analyisis, median captures more of the more common transaction amounts, compared to the mean
df.drop(columns=["global_avg_amount", "global_amount_ratio", "global_log_amount_ratio", "global_median_amount"], inplace=True)

# From Permutation graphs log values tend to do better
df.drop(columns=["customer_avg_amount_ratio", "merchant_avg_amount_ratio", "global_median_amount_ratio"], inplace=True)

# These columns are only used to build std during inference
df.drop(columns=["customer_amount_sq_sum", "customer_M2_amount", "merchant_amount_sq_sum","merchant_M2_amount", "global_amount_sq_sum", "global_M2_amount"], inplace=True)

# Index isnt perserved in the DB
df.drop(columns=["global_transaction_count"], inplace=True)

### Sort columns into categories
Split each column by their characteristic (Categorical, Coninous, Binary)

In [ ]:
cat_cols, cont_cols, binary_cols = [], [], []

for col in df.columns:
    if df[col].dtype == "str":
        cat_cols.append(col)
    else:
        if df[col].nunique() > 2:
            cont_cols.append(col)
        else:
            binary_cols.append(col)


print("Categorical Columns:", cat_cols)
print("Continuous Columns:", cont_cols)
print("Binary Columns:", binary_cols)

Split further into which features link to local/global features


In [ ]:
feature_dict = {"global" : [], "customer": [], "merchant": [], "other": []}

for col in cont_cols:
    if col.startswith("global_"):
        feature_dict["global"].append(col)
    elif col.startswith("customer"):
        feature_dict["customer"].append(col)
    elif col.startswith("merchant"):
        feature_dict["merchant"].append(col)
    else:
        feature_dict["other"].append(col)

print("Global Features:", feature_dict["global"])
print("Customer Features:", feature_dict["customer"])
print("Merchant Features:", feature_dict["merchant"])
print("Other Features:", feature_dict["other"])

### Check class distribution
If classes are imbalanced, avoid using Accuracy metric. Use F1 Score, Percision and Recall instead to measure model preformance.

In [ ]:
# count values of classifier column
fraud_counts = df['fraud'].map({0: "No", 1: "Yes"}).value_counts()

bin_dict = {"age": [], "gender": []}
for col in binary_cols:
    if col.startswith("age"):
        bin_dict["age"].append(col)
    elif col.startswith("gender"):
        bin_dict["gender"].append(col)

age_counts = df[bin_dict["age"]].sum()
gender_counts = df[bin_dict["gender"]].sum()

print("Fraud Counts:", fraud_counts)
print("\nAge Counts:\n", age_counts)
print("\nGender Counts:\n", gender_counts)

In [ ]:
def pie_with_arrows(ax, data, title):
    wedges, _ = ax.pie(data, startangle=90)
    ax.set_title(title)

    for i, wedge in enumerate(wedges):
        angle = (wedge.theta2 + wedge.theta1) / 2
        x = np.cos(np.deg2rad(angle))
        y = np.sin(np.deg2rad(angle))

        label = f"{data.index[i]}: {data.iloc[i] / data.sum() * 100:.1f}%"

        ax.annotate(
            label,
            xy=(x, y),
            xytext=(1.2 * np.sign(x), 1.2 * y),
            arrowprops=dict(arrowstyle="-"),
            ha="left" if x > 0 else "right"
        )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 10))

pie_with_arrows(axes[0], fraud_counts, "Fraud Distribution")
pie_with_arrows(axes[1], age_counts, "Age Distribution")
pie_with_arrows(axes[2], gender_counts, "Gender Distribution")

plt.tight_layout()
plt.show()

### Variance Calculation
Looks at the number of unique values in a column if its just 1 drop it. No variance means the model wouldnt have any distinctive information for the model


In [ ]:
cols_to_drop = []
for col in df.columns:
   unum = df[col].nunique()

   print(f"Unique numbers of {col}s:", unum)

   if unum == 1:
      cols_to_drop.append(col)

print("\nDropping columns due to constant values:", cols_to_drop)

for col in cols_to_drop:
   cat_cols.remove(col)

df.drop(columns=cols_to_drop, inplace=True)

### Look at transaction amounts
See how tranasaction amount distribute across the dataset and how it correlates with fraud


In [ ]:
mean = df["customer_amount"].mean()
median = df["customer_amount"].median()
q99 = df["customer_amount"].quantile(0.99)

fraud_values = df["customer_amount"][df["fraud"] == 1]
normal_values = df["customer_amount"][df["fraud"] == 0]

fig, axes = plt.subplots(3, 1,figsize=(40, 18))

_, bin_edges, _ = axes[0].hist(df["customer_amount"], bins=50, range=(df["customer_amount"].min(),q99))

axes[0].set_title("Customer Amount Distribution")
axes[0].set_xticks(bin_edges)
axes[0].set_xlabel("Customer Amount")
axes[0].set_ylabel("Frequency")

axes[0].axvline(mean, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean:.2f}')
axes[0].axvline(median, color='green', linestyle='solid', linewidth=2, label=f'Median: {median:.2f}')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.7)

# Density captures the probability distribution, probability = max_bin_width * height(density)
axes[1].hist(normal_values, bins=50, alpha=0.6, label='Normal', density=True, range=(normal_values.min(), q99))
axes[1].hist(fraud_values, bins=50, label='Fraud', color="orange", density=True, range=(fraud_values.min(), q99))

axes[1].set_title("Customer Amount Distribution by Fraud Status")
axes[1].set_xticks(bin_edges)
axes[1].set_xlabel("Customer Amount")
axes[1].set_ylabel("Probability Density")

axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.7)

axes[2].boxplot(
    [
        normal_values,
        fraud_values
    ],
    vert=False,
    showfliers=False,
    labels=["Normal", "Fraud"]
)
axes[2].set_title("Customer Amount Box Plot")


Fraud rate probability is roughly uniform across the distirbution, probably a cause to why most models perfer fraud rate compared to using any of the features derived from amounts as they are mostly just noise

### Correlation Calculation and Visualisation
Sees if any two features have a linear relationship. Two features that are related and grow/decrease linearly can point towards a feature that can be dropped so the model isn't being fed the same information again.

In [ ]:
cor_cof= df[cont_cols + ["fraud"]].corr()
mask = np.triu(np.ones_like(cor_cof, dtype=bool), k=1) # Correlation is symmetric, so we only need to show one triangle of the matrix

plt.figure(figsize=(20, 16))
sns.heatmap(cor_cof, mask=mask, annot=True, vmin=-1, vmax=1, cbar_kws={"aspect": 40}, cmap="viridis", fmt=".2f", xticklabels=cor_cof.columns, yticklabels=cor_cof.columns, square=True, annot_kws={"size": 10})
plt.title("Correlation between features")
plt.tight_layout()
plt.show()


For example, we can see that merchant_log_amount_ratio and merchant_z_score are highly correlated in the dataset and they both roughly capture the same meaning just in different numerical forms, It could be dropped especially in the case of the classical models, however it was decided to be kept due to how the Neural-Nets might use them as they can combine the customer/merchant features allowing the models to learn more nuanced behavioural patterns that may improve fraud detection performance.

In [ ]:
# Finds the features that are most correlated to the target(Fraud)
target_corr = cor_cof['fraud']
target_corr_cont = target_corr[cont_cols] # Ignores binarry features as they are not useful for scatter plots

top2_vals = target_corr_cont.abs().sort_values(ascending=False)[:2] #Find the top 2 features most correlated with fraud
print(top2_vals)

top2_features = top2_vals.index.to_list()

### Plotting the two most correlated features
A plot of features to see how fraud and non-fraud strucutre is like in a basic linear space. The plot gives information on how trivial the problem is, where if the two categories seperate well simpler models(Linear/Logistic) would perform better or if they are highly mixed a stronger model(NNs/Trees) could perform better.

In [ ]:
f1, f2 = top2_features

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=df[f1],
    y=df[f2],
    hue=df['fraud'],
    alpha=0.7,
)

plt.show()

### Plot Merchant Fraud Rate


In [ ]:
merchant_freq = df["merchant"].value_counts()

merchant_last = df.groupby("merchant").last()
fraud_rate = merchant_last.loc[merchant_freq.index, "merchant_fraud_rate"]

fraud_counts = merchant_freq * fraud_rate
non_fraud = merchant_freq * (1 - fraud_rate)

fig, ax = plt.subplots(1, 1, figsize=(20, 8))
ax.bar(merchant_freq.index.astype(str), non_fraud, label="Non-Fraud")
ax.bar(merchant_freq.index.astype(str), fraud_counts, bottom=non_fraud, label="Fraud", color="red")
    
ax.set_title("Transaction Count per Merchant (Fraud Split)")
ax.set_yscale("log")
ax.set_xlabel("Merchant")
ax.set_ylabel("Log(Number of Transactions)")
ax.set_xticklabels(merchant_freq.index.astype(str), rotation=90)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.show()

Looks at if a Merchant has a more than one category for the GNN model

In [ ]:
category_cols = [c for c in df.columns if c.startswith("category_")]

df.groupby("merchant")[category_cols].max().sum(axis=1).sort_values(ascending=False)